# Restore Vivienne's photos — free GPU

Runs **CodeFormer** face-restoration over all the full-res photos on Google's free GPU, then hands you back one `.zip`.

**How to use:**
1. Top menu: **Runtime → Change runtime type → T4 GPU** → Save.
2. **Runtime → Run all** (or press the play button on each cell, top to bottom).
3. Wait ~20–30 min. The last cell downloads `restored.zip` to your computer.
4. Send me that zip (or drop it in the repo) and I'll wire the photos in.

Nothing here touches your originals — it reads the public repo and writes new files only.

In [ ]:
# 1) Confirm we actually have a GPU (if this errors, do Runtime → Change runtime type → T4 GPU)
import torch
assert torch.cuda.is_available(), 'No GPU. Set Runtime → Change runtime type → T4 GPU, then Run all again.'
print('GPU ready:', torch.cuda.get_device_name(0))

In [ ]:
# 2) Get CodeFormer + install dependencies (a couple minutes)
%cd /content
!rm -rf CodeFormer
!git clone https://github.com/sczhou/CodeFormer.git
%cd /content/CodeFormer
!pip install -q -r requirements.txt
!pip install -q basicsr facexlib gfpgan realesrgan
!python basicsr/setup.py develop
# fetch the pretrained weights
!python scripts/download_pretrained_models.py facelib
!python scripts/download_pretrained_models.py CodeFormer

In [ ]:
# 3) Pull the photos from the public repo into an inputs/ folder
%cd /content
!rm -rf family inputs && git clone --depth 1 https://github.com/MarcBittner/family.git
import os, shutil
os.makedirs('/content/inputs', exist_ok=True)
src = '/content/family/montage/full'
n = 0
for f in sorted(os.listdir(src)):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        shutil.copy(os.path.join(src, f), os.path.join('/content/inputs', f))
        n += 1
print('photos to restore:', n)

In [ ]:
# 4) Run the restoration on the GPU. -w 0.7 balances fidelity vs. quality; --face_upsample sharpens faces.
%cd /content/CodeFormer
!python inference_codeformer.py -w 0.7 --face_upsample --bg_upsampler realesrgan \
    -i /content/inputs -o /content/results

In [ ]:
# 5) Rename to the site's convention (0001.jpg → p0001.jpg), downsize for web, and zip
%cd /content
import os, subprocess
fin = '/content/results/final_results'
out = '/content/restored'
os.makedirs(out, exist_ok=True)
cnt = 0
for f in sorted(os.listdir(fin)):
    if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    stem = os.path.splitext(f)[0]
    dst = os.path.join(out, f'p{stem}.jpg')
    # keep crisp but reasonable for the web
    subprocess.run(['convert', os.path.join(fin, f), '-resize', '1400x1400>', '-quality', '88', dst])
    cnt += 1
print('restored & resized:', cnt)
!cd /content && zip -q -r restored.zip restored
print('zip ready:', os.path.getsize('/content/restored.zip') // (1024*1024), 'MB')

In [ ]:
# 6) Download restored.zip to your computer
from google.colab import files
files.download('/content/restored.zip')